<a href="https://colab.research.google.com/github/ArcVielLouvent/PJK-RM119-CareerMatch-AI/blob/main/PJK_RM119_CareerMatch_AI_Workspace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workspace Kolaborasi: CareerMatch AI
**Tim:** PJK-RM119

**Deskripsi:** Notebook utama untuk pengembangan sistem rekomendasi ATS CV menggunakan NLP.
Notebook ini dibagi berdasarkan target mingguan dan pembagian peran (Role) masing-masing anggota.

---
## INISIALISASI & SETUP LINGKUNGAN (Global)
Bagian ini berisi instalasi library dan import modul yang akan digunakan oleh seluruh tim. Pastikan untuk menjalankan cell ini pertama kali, dan update cell ini sesuai dengan kebutuhan masing masing.

In [ ]:
# Install library eksternal utama
!pip install kagglehub pypdf2 spacy nltk pandas numpy scikit-learn mlflow tqdm

# Unduh model bahasa Inggris untuk SpaCy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2

In [ ]:
# Update cell ini sesuai kebutuhan
# Import seluruh library yang dibutuhkan
import os
import pandas as pd
import numpy as np
import re
import spacy
import nltk
import kagglehub
from PyPDF2 import PdfReader
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import mlflow
from google.colab import files
import io

# Setup NLTK
nltk.download('stopwords')
nltk.download('punkt')

print("Setup Environment Selesai!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Setup Environment Selesai!


---
## MINGGU 1: PENGUMPULAN DATA & EKSPLORASI

### Tugas 1.1: Ekstraktor Dokumen PDF
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Membuat fungsi untuk mengekstrak teks mentah dari file PDF CV (Format ATS-Friendly).

In [ ]:
def extract_pdf_text(pdf_path):
    """
    Ekstraksi teks mentah dari PDF dengan penanganan error kelas produksi.
    Fungsi ini bertanggung jawab mengubah file binary PDF menjadi raw text.
    """
    try:
        reader = PdfReader(pdf_path)
        raw_text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                raw_text += extracted + " "

        if not raw_text.strip():
            return None, "Dokumen kosong atau berbasis gambar (Tidak ATS-Friendly)."

        return raw_text, "Success"

    except Exception as e:
        return None, str(e)

### Tugas 1.2: Akuisisi Dataset Lowongan Kerja (Kaggle)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Mencari dataset *Job Postings* atau yang berkaitan yang relevan di Kaggle dan memuatnya ke dalam Pandas DataFrame.

> **Saran:** Hindari mengunduh manual. Gunakan library `kagglehub` untuk langsung menarik data ke dalam memori.

In [ ]:
# Tulis kode download dataset Kaggle dan load ke dalam DataFrame (df_jobs) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Download dataset dari Kaggle
path = kagglehub.dataset_download("madhab/jobposts")

print("Path to dataset files:", path)

100%|██████████| 14.0M/14.0M [00:00<00:00, 53.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/madhab/jobposts/versions/1


In [ ]:
# Melihat file yang tersedia
files = os.listdir(path)

print("Daftar file dalam dataset:")
for file in files:
    print(file)

Daftar file dalam dataset:
screenshot.jpg
data job posts.csv


In [ ]:
# Ganti nama file sesuai file CSV yang muncul
csv_path = os.path.join(path, "data job posts.csv")

# Load dataset
df_jobs = pd.read_csv(csv_path)

### Tugas 1.3: Exploratory Data Analysis (EDA)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Melakukan pengecekan karakteristik data awal. Cek total baris, nama kolom, jumlah *missing values*, dan lihat sekilas isi kolom deskripsi pekerjaan.

In [ ]:
# Tulis kode untuk EDA (df.head, df.info, df.isna().sum()) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Menampilkan 5 data pertama
df_jobs.head()

,jobpost,date,Title,Company,AnnouncementCode,Term,Eligibility,Audience,StartDate,Duration,...,Salary,ApplicationP,OpeningDate,Deadline,Notes,AboutC,Attach,Year,Month,IT
0,AMERIA Investment Consulting Company\r\nJOB TI...,"Jan 5, 2004",Chief Financial Officer,AMERIA Investment Consulting Company,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,"To apply for this position, please submit a\r\...",NaN,26 January 2004,NaN,NaN,NaN,2004,1,False
1,International Research & Exchanges Board (IREX...,"Jan 7, 2004",Full-time Community Connections Intern (paid i...,International Research & Exchanges Board (IREX),NaN,NaN,NaN,NaN,NaN,3 months,...,NaN,Please submit a cover letter and resume to:\r\...,NaN,12 January 2004,NaN,The International Research & Exchanges Board (...,NaN,2004,1,False
2,Caucasus Environmental NGO Network (CENN)\r\nJ...,"Jan 7, 2004",Country Coordinator,Caucasus Environmental NGO Network (CENN),NaN,NaN,NaN,NaN,NaN,Renewable annual contract\r\nPOSITION,...,NaN,Please send resume or CV toursula.kazarian@......,NaN,20 January 2004\r\nSTART DATE: February 2004,NaN,The Caucasus Environmental NGO Network is a\r\...,NaN,2004,1,False
3,Manoff Group\r\nJOB TITLE: BCC Specialist\r\n...,"Jan 7, 2004",BCC Specialist,Manoff Group,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Please send cover letter and resume to Amy\r\n...,NaN,23 January 2004\r\nSTART DATE: Immediate,NaN,NaN,NaN,2004,1,False
4,Yerevan Brandy Company\r\nJOB TITLE: Software...,"Jan 10, 2004",Software Developer,Yerevan Brandy Company,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Successful candidates should submit\r\n- CV; \...,NaN,"20 January 2004, 18:00",NaN,NaN,NaN,2004,1,True


In [ ]:
# Info dataset
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19001 entries, 0 to 19000
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   jobpost           19001 non-null  object
 1   date              19001 non-null  object
 2   Title             18973 non-null  object
 3   Company           18994 non-null  object
 4   AnnouncementCode  1208 non-null   object
 5   Term              7676 non-null   object
 6   Eligibility       4930 non-null   object
 7   Audience          640 non-null    object
 8   StartDate         9675 non-null   object
 9   Duration          10798 non-null  object
 10  Location          18969 non-null  object
 11  JobDescription    15109 non-null  object
 12  JobRequirment     16479 non-null  object
 13  RequiredQual      18517 non-null  object
 14  Salary            9622 non-null   object
 15  ApplicationP      18941 non-null  object
 16  OpeningDate       18295 non-null  object
 17  Deadline    

In [ ]:
# Mengecek data kosong
df_jobs.isna().sum().sort_values(ascending=False)

,0
Audience,18361
AnnouncementCode,17793
Attach,17442
Notes,16790
Eligibility,14071
Term,11325
Salary,9379
StartDate,9326
Duration,8203
AboutC,6531


---
## MINGGU 2: PRA-PEMROSESAN DATA (PREPROCESSING)

### Tugas 2.1: Data Cleaning (Pembersihan Dataset)
**PIC:** Aisyah (Data Analyst)

**Deskripsi Tugas:** Menghapus data duplikat dan baris yang kolom *Job Description/Requirements*-nya kosong (*missing values*), karena data kosong akan merusak perhitungan NLP.

In [ ]:
# Tulis kode pembersihan DataFrame (dropna, drop_duplicates) di sini dan tambahkan cell kode sesuai dengan kebutuhan

# Memilih kolom yang relevan
clean_df = df_jobs[[
    "Title",
    "JobDescription",
    "JobRequirment",
    "RequiredQual"
]]

In [ ]:
# Mengecek missing values
clean_df.isna().sum()

,0
Title,28
JobDescription,3892
JobRequirment,2522
RequiredQual,484


In [ ]:
# Menghapus baris yang kosong
clean_df = clean_df.dropna(subset=[
    "Title",
    "JobDescription",
    "JobRequirment",
    "RequiredQual"
])

In [ ]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13124 entries, 0 to 19000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           13124 non-null  object
 1   JobDescription  13124 non-null  object
 2   JobRequirment   13124 non-null  object
 3   RequiredQual    13124 non-null  object
dtypes: object(4)
memory usage: 512.7+ KB


In [ ]:
# Menghapus data duplikat
clean_df = clean_df.drop_duplicates()

In [ ]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12379 entries, 0 to 19000
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           12379 non-null  object
 1   JobDescription  12379 non-null  object
 2   JobRequirment   12379 non-null  object
 3   RequiredQual    12379 non-null  object
dtypes: object(4)
memory usage: 483.6+ KB


In [ ]:
clean_df.head()

,Title,JobDescription,JobRequirment,RequiredQual
0,Chief Financial Officer,AMERIA Investment Consulting Company is seekin...,- Supervises financial management and administ...,"To perform this job successfully, an\r\nindivi..."
2,Country Coordinator,Public outreach and strengthening of a growing...,- Working with the Country Director to provide...,"- Degree in environmentally related field, or ..."
3,BCC Specialist,The LEAD (Local Enhancement and Development fo...,- Identify gaps in knowledge and overseeing in...,"- Advanced degree in public health, social sci..."
13,"Community Development, Capacity Building and C...",Food Security Regional Cooperation and Stabili...,- Assist the Tavush Marz communities and commu...,- Higher Education and/or professional experie...
17,Country Economist (NOB),The United Nations Development Programme in Ar...,The incumbent under direct supervision of UNDP...,- Minimum Masters Degree in Economics;\r\n- Mi...


### Tugas 2.2: NLP Text Preprocessing Pipeline
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Membuat satu fungsi terpusat (`clean_text`) untuk menyeragamkan teks (huruf kecil, hapus URL, hapus karakter khusus) dan melakukan Tokenisasi, Stopwords Removal, serta Lemmatization menggunakan SpaCy/NLTK. Fungsi ini akan di-apply ke dataset lowongan kerja dan juga CV pengguna nanti.

In [ ]:
# Tulis kode fungsi clean_text() dan aplikasikan ke kolom df_jobs di sini

# Inisialisasi model NLP spaCy (Ringan)
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# Daftar stopwords umum untuk menyingkirkan noise pada CV dan Job Description
universal_stopwords = {
    'summary', 'experience', 'education', 'skill', 'skills', 'qualification',
    'project', 'projects', 'certification', 'honor', 'award', 'contact',
    'email', 'phone', 'location', 'github', 'linkedin', 'profile', 'work',
    'status', 'expected', 'current', 'gpa', 'cumulative', 'university',
    'student', 'degree', 'diploma', 'bachelor', 'master'
}
stop_words = set(nltk.corpus.stopwords.words('english')).union(universal_stopwords)

def clean_text(raw_text):
    """
    Fungsi terpusat untuk membersihkan dan menstandarisasi teks (CV maupun Dataset).
    Meliputi case-folding, pembersihan karakter non-alfanumerik, dan lemmatization.
    """
    if not isinstance(raw_text, str) or not raw_text:
        return ""

    # 1. Lowercase
    text = raw_text.lower()

    # 2. Hapus URL dan Email
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)

    # 3. Hanya pertahankan huruf dan angka (alfanumerik) untuk menjaga istilah IT (e.g., HTML5)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 4. Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Lemmatization & Stopwords Removal
    doc = nlp(text)
    purified_tokens = [
        token.lemma_ for token in doc
        if token.lemma_ not in stop_words and len(token.lemma_) > 1 and not token.text.isnumeric()
    ]

    return " ".join(purified_tokens)

In [ ]:
# ==========================================
# EKSEKUSI PIPELINE NLP KE DATASET
# ==========================================

print("1. Menggabungkan kolom teks menjadi satu dokumen utuh...")
# Menggabungkan Title, Description, Requirement, dan Qualification
clean_df['Combined_Text'] = clean_df['Title'] + " " + \
                            clean_df['JobDescription'].fillna('') + " " + \
                            clean_df['JobRequirment'].fillna('') + " " + \
                            clean_df['RequiredQual'].fillna('')

print("2. Menerapkan Arcsviel NLP Protocol (clean_text) ke seluruh dataset...")
# Mengaktifkan progress bar dari tqdm untuk pandas
tqdm.pandas(desc="Membersihkan Teks")

# Menerapkan fungsi clean_text ke kolom gabungan
clean_df['Cleaned_Text'] = clean_df['Combined_Text'].progress_apply(clean_text)

print("\n PRA-PEMROSESAN SELESAI ")
# Melihat hasil akhir
clean_df[['Combined_Text', 'Cleaned_Text']].head()

---
## MINGGU 3: PEMBANGUNAN MODEL (MACHINE LEARNING)

### Tugas 3.1: Ekstraksi Fitur & Matriks Kemiripan
**PIC:** Armand (Lead AI Engineer)

**Deskripsi Tugas:** Mengubah teks bersih menjadi vektor numerik menggunakan algoritma **TF-IDF**. Kemudian membangun fungsi untuk menghitung **Cosine Similarity** antara vektor CV dengan matriks vektor lowongan kerja. Terapkan MLflow tracking jika bereksperimen dengan parameter (max_features, n-grams).

---
## MINGGU 4: PENGEMBANGAN ANTARMUKA (DEPLOYMENT)

### Tugas 4.1: Desain & Logika Aplikasi Streamlit
**PIC:** Islahul (UI/UX) dibantu oleh Armand (Integrasi Model)

**Deskripsi Tugas:** Merancang tata letak aplikasi web (Front-end). *Catatan: Kode untuk Streamlit biasanya ditulis di file `app.py` terpisah, namun cell ini dapat digunakan untuk membuat prototipe UI sederhana atau fungsi render.*

In [ ]:
# Area eksperimen prototipe logika Streamlit UI

---
## MINGGU 5: PENGUJIAN & QUALITY ASSURANCE

### Tugas 5.1: Pengujian Batas (Edge Cases)
**PIC:** Faber (QA Tester)

**Deskripsi Tugas:** Menguji sistem dengan mengunggah berbagai jenis skenario (CV kosong, CV penuh gambar/tidak terbaca PyPDF2, CV bahasa campuran, CV dengan format aneh).

> **Saran:**
> Dokumentasikan setiap *error* yang muncul di cell ini dan laporkan ke LeadAI Engineer agar fungsi Python-nya bisa diperbaiki (*error handling*).

In [ ]:
# Tulis kode pengujian atau buat log testing menggunakan skenario khusus di sini